<a href="https://colab.research.google.com/github/PavloMytsko/lab3-ocr-variant10/blob/main/notebooks/Lab3-MytskoPavlo-Variant10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Лабораторна робота №3
## Побудова рішення розпізнавання тексту із застосуванням ШІ

**Дисципліна:** Технології ШІ в інженерії даних
**Виконав:** Мицко Павло, група ПЗ-46
**Варіант:** 10
**Завдання:** порівняння Tesseract та EasyOCR для розпізнавання тексту на чеках

## 1. Personal Information and Reproducibility



In [4]:
# Завдання 1. Персональні параметри та відтворюваність

STUDENT = "Мицко Павло"
GROUP = "ПЗ-46"
VARIANT = 10
STUDENT_ID = 534

SEED = 1000 * VARIANT + STUDENT_ID

print("Студент:", STUDENT)
print("Група:", GROUP)
print("Варіант:", VARIANT)
print("STUDENT_ID:", STUDENT_ID)
print("SEED:", SEED)

assert SEED == 10534

print("Персональні параметри задано правильно.")

Студент: Мицко Павло
Група: ПЗ-46
Варіант: 10
STUDENT_ID: 534
SEED: 10534
Персональні параметри задано правильно.


## 2. Installing and Importing Libraries

In [1]:
# Завдання 2. Встановлення системного OCR та Python-бібліотек

!sudo apt-get update -qq
!sudo apt-get install -y -qq tesseract-ocr tesseract-ocr-ukr tesseract-ocr-eng

%pip install -q pytesseract rapidfuzz easyocr datasets jiwer

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package tesseract-ocr-ukr.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-ukr_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-ukr (1:4.1.0-2) ...
Setting up tesseract-ocr-ukr (1:4.1.0-2) ...


In [2]:
# Перевірка встановлення Tesseract та мовних моделей

!tesseract --version
!tesseract --list-langs

tesseract 5.3.4
 leptonica-1.82.0
  libgif 5.2.1 : libjpeg 8d (libjpeg-turbo 2.1.5) : libpng 1.6.43 : libtiff 4.5.1 : zlib 1.3 : libwebp 1.3.2 : libopenjp2 2.5.0
 Found AVX2
 Found AVX
 Found FMA
 Found SSE4.1
 Found OpenMP 201511
 Found libarchive 3.7.2 zlib/1.3 liblzma/5.4.5 bz2lib/1.0.8 liblz4/1.9.4 libzstd/1.5.5
 Found libcurl/8.5.0 OpenSSL/3.0.13 zlib/1.3 brotli/1.1.0 zstd/1.5.5 libidn2/2.3.7 libpsl/0.21.2 (+libidn2/2.3.7) libssh/0.10.6/openssl/zlib nghttp2/1.59.0 librtmp/2.3 OpenLDAP/2.6.10
List of available languages in "/usr/share/tesseract-ocr/5/tessdata/" (3):
eng
osd
ukr


In [5]:
# Імпорт бібліотек

import os
import re
import json
import glob
import time
import random
import hashlib
import platform
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageDraw, ImageFont, ExifTags
from scipy import stats
from rapidfuzz.distance import Levenshtein
from datasets import load_dataset

import pytesseract
import easyocr
import torch

random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

print("Python:", platform.python_version())
print("OpenCV:", cv2.__version__)
print("Tesseract:", pytesseract.get_tesseract_version())
print("EasyOCR:", easyocr.__version__)
print("GPU доступний:", torch.cuda.is_available())
print("Бібліотеки імпортовано успішно.")

Python: 3.13.18
OpenCV: 5.0.0
Tesseract: 5.3.4
EasyOCR: 1.7.2
GPU доступний: False
Бібліотеки імпортовано успішно.


## 3. Hypotheses Before Experiments


### Гіпотези до проведення експериментів

Гіпотези сформульовано до запуску OCR-експериментів. Після отримання результатів кожна гіпотеза буде оцінена як підтверджена або спростована.

#### Гіпотеза 1. Якість на зображеннях без повороту

На синтетичних зображеннях без повороту Tesseract матиме CER не більше 5%, а EasyOCR — не більше 8%. Очікується, що на чистому друкованому тексті Tesseract буде точнішим, оскільки зображення мають просте розташування тексту та високу контрастність.

#### Гіпотеза 2. Стійкість до повороту

При повороті на 10° CER Tesseract перевищить 15%, тоді як CER EasyOCR не перевищить 12%. Очікується, що EasyOCR буде стійкішим до значних поворотів завдяки нейромережевій детекції текстових областей. Передбачається, що криві якості інструментів перетнуться в діапазоні від 4° до 8°.

#### Гіпотеза 3. Вплив вирівнювання нахилу

Застосування deskew для зображень, повернутих на 10° і 15°, зменшить середній CER Tesseract щонайменше на 30%, а EasyOCR — щонайменше на 15%. На власних фотографіях чеків під кутом deskew має зменшити середній CER щонайменше на 25% порівняно з необробленими фотографіями.

## 4. Text Normalization and Metrics

## 5. Synthetic Dataset Creation

## 6. Open Dataset CORD v2

## 7. Own Images and Ground Truth

## 8. Image Preprocessing

## 9. Tesseract and EasyOCR

## 10. Synthetic Data Experiment


## 11. Open Dataset Experiment

## 12. Own Images Experiment

## 13. Ablation Study

## 14. Robustness and Confidence Analysis

## 15. Error Analysis

## 16. Deskew Experiment

## 17. Date and Amount Extraction

## 18. Saving Results to JSON

## 19. Conclusions

## 20. AI Usage Declaration